# سبق 18 (پیروی): رسیدیں جو ثابت کرتی ہیں کہ ایک *انسانی* نے کارروائی کی منظوری دی

یہ سبق ثابت کرتا ہے کہ **ایجنٹ** نے کیا کیا اور **گیٹ** نے کیا فیصلہ کیا۔ یہ نوٹ بک وہ گم شدہ حصہ شامل کرتی ہے: ثبوت کہ ایک **مخصوص انسان** نے **عین** کارروائی کی منظوری دی — ایک الگ، انسان کے قبضے میں دستخط جو مکمل کینونیکل کارروائی پر ہوتا ہے، اور آف لائن تصدیق کیا جاتا ہے۔

یہاں دونوں دستاویزات سبق کی رسیدوں کی **اسی لفافہ شکل** استعمال کرتی ہیں: ایک فلیٹ پیلوڈ جس میں `type` فیلڈ ہوتی ہے، جو Ed25519 کے ذریعے براہ راست کینونیکل JCS بائٹس پر دستخط شدہ ہوتی ہے، اور اس کے ساتھ ایک ساختہ `signature` آبجیکٹ منسلک ہوتا ہے (جو دستخط شدہ بائٹس سے خارج ہے)۔ منظوری رسید ایک نیا `type` (`human.approval.v1`) ہے جو کارروائی کی قسم کے ساتھ ساتھ ہوتا ہے، اس لیے ایک `verify_chain` دونوں قسم کی دستاویزات کو ایک ہی کوڈ راستے سے کور کرتا ہے جو آپ نے مرکزی نوٹ بک میں بنایا تھا۔ یہ انسانی منظوری رسید ایک تعلیمی ترکیب ہے جو یہاں بیان کی گئی ہے، نہ کہ draft-farley-acta-signed-receipts کی طرف سے کوئی رسید کی قسم۔

مرکزی نوٹ بک میں ڈیمو ویریفائر کے مقابلے میں ایک ارادی اپ گریڈ: یہاں ویریفائر `signature.key_id` کو رسید کے اندر لے جانے والی پبلک کی پر اعتماد کرنے کی بجائے **پِن کی گئی کی رجسٹری** کے خلاف حل کرتا ہے۔ یہ وہ پیداواری رویہ ہے جو سبق کی اپنی چیک لسٹ تجویز کرتی ہے ("تصدیقی پبلک کی شائع کریں")، اور یہی جعل سازی کو انکار بناتا ہے نہ کہ اپنا کی لانے کا راستہ۔

یہ نوٹ بک جو قاعدہ سکھاتی ہے: **ایک دستخط شدہ منظوری خود میں اختیار نہیں ہوتی۔** اختیار صرف تب موجود ہوتا ہے جب منظوری رسید اور کارروائی رسید اطلاق کے وقت اب بھی ایک ہی کینونیکل کارروائی سے بندھے ہوں، ایک پالیسی ورژن، کی، اور ایکسپائری کے تحت جو اب بھی فعال ہوں، اور ایسی منظوری جو پہلے استعمال نہ کی گئی ہو۔ ہر ناکامی ایک **مخصوص وجہ** کے ساتھ انکار کرتی ہے، تاکہ آپ *اختیار کا پرانا ہونا* اور *کارروائی میں تبدیلی* کو الگ پہچان سکیں۔


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## درست عمل

منظوری کی اکائی **معیاری عمل کی شے** ہے — کوئی مبہم لیبل نہیں جیسے "ریفنڈ کی منظوری،" بلکہ درست، مکمل طور پر مخصوص عمل۔ پورے شے پر دستخط کرنا (اور اس سے ایک خلاصہ تیار کرنا) یہی ہمیں بعد میں ثابت کرنے دیتا ہے کہ انسان نے *یہی* منظور کیا اور کچھ نہیں۔


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## ایک لفافہ، دو اتھارٹیز

ہر رسید سبق کا لفافہ ہے: ایک فلیٹ پیلوڈ جس میں ایک `type` فیلڈ ہوتی ہے، اور ایک `signature` آبجیکٹ (`alg`, `sig`, `key_id`) ہوتا ہے جو دستخط شدہ بائٹس کا حصہ **نہیں** ہوتا۔ `verify_envelope` دونوں قسم کی رسیدوں کے لیے مشترکہ ساختی اور دستخطی چیک ہے؛ جس **پن کی رجسٹری** میں یہ `signature.key_id` کو حل کرتا ہے وہی اتھارٹیز کو الگ رکھتا ہے:

- **approval receipt** (`human.approval.v1`) — نامزد منظوری دہندہ، مکمل کینونیکل عمل **اور اس کا ڈائجسٹ**, `policy_version`, اجرا + ختم ہونے کے ٹائم اسٹیمپس۔ ایک بار استعمال کو چین کی سطح پر ٹریک کیا جاتا ہے۔
- **action receipt** (`agent.action.v1`) — ایجنٹ کی شناخت، `run_id`, وہی کینونیکل عمل **ڈائجسٹ**, عمل درآمد کا نتیجہ + ٹائم اسٹیمپ، اور `parent_approval_ref`: منظوری کی `receipt_hash`, سبق کی چین میں `previous_receipt_hash` کی طرح کا کنونشن۔

مشترکہ `action_digest` فیلڈ وہ بندھن ہے جس پر جوائن انحصار کرتا ہے۔ `key_id` صرف دستخطی آبجیکٹ میں ایک تلاش کے اشارے کے طور پر موجود ہوتا ہے: اسے کسی مختلف پن کی طرف موڑنے سے دستخطی چیک ناکام ہو جاتا ہے، اس لیے یہ کچھ ظاہر نہیں کرتا۔


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the canonical JCS
    bytes directly; the signature object is NOT part of the signed bytes."""
    canonical = canonicalize(payload)
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(canonical).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized and passed directly to Ed25519.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        canonical = canonicalize(payload)
        VerifyKey(b64url_decode(pub)).verify(canonical, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: جہاں بندھن درحقیقت فیصلہ کیا جاتا ہے  

`verify_chain` دو دستخطی چیکز پر ایک سہولت لپیٹ نہیں ہے۔ یہ وہ واحد جگہ ہے جہاں مشترکہ معیاری `action_digest`، منظوری کی پالیسی/چابی/میعاد ختم ہونے کی **تازگی**، اور منظوری کے **ایک بار استعمال** کو ایک ساتھ چیک کیا جاتا ہے، اس عمل کے خلاف جو *ابhi انجام دیا جا رہا ہے*۔  

ہر ناکامی ایک **منفرد وجہ** کے ساتھ انکار کرتی ہے، تاکہ انکار پڑھنے والا بتا سکے کہ آیا اختیار پرانی ہو گئی ہے (پالیسی بدلی، چابی گھمائی گئی، منظوری کی میعاد ختم ہو گئی، منظوری استعمال ہو چکی ہے) یا انجام دی جانے والی کارروائی نے ابھی بھی درست منظوری کے تحت (ڈائجسٹ کی تبدیلی) تبدیلی کی ہے۔  


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## پابندی کیا پکڑتی ہے

نیچے ہر کیس **ایک ممتاز وجہ** کے ساتھ **بند** ہوتا ہے۔ پہلا گروپ کلاسیکی سیٹ ہے (چالاکی، الجھ گیا نمائندہ، دوبارہ چلانا، اختیار پر جعلسازی، خراب اندراج)۔ دوسرا گروپ جو خاصیت کو محض دعویٰ کرنے کی بجائے حقیقت بناتا ہے وہ ہے:

- **پرانی اختیار** — دستخط اب بھی درست ہے، لیکن پالیسی ورژن بدل چکا ہے، منظوری دینے والا کلید پنڈ رجسٹری سے نکال دیا گیا، یا منظوری عمل درآمد سے پہلے ختم ہوگئی؛
- **ڈائجسٹ کی جگہ تبدیلی** — ایک درست دستخط شدہ کارروائی رسید جس کا `parent_approval_ref` ایک *حقیقی* منظوری کی طرف اشارہ کرتا ہے، لیکن اس منظوری کا رسمی کارروائی ڈائجسٹ اس کارروائی سے میل نہیں کھاتا جو اصل میں عمل میں لائی جا رہی ہے۔


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## یہ کیا ثابت کرتا ہے — اور کیا نہیں کرتا

**ثابت کرتا ہے:** ایک نامزد انسان نے *اسی عین قانونی اقدام* کی منظوری دی (مکمل اقدام + ڈائجسٹ، ایک کلید سے دستخط شدہ جو pinned رجسٹری سے حل کی گئی)، اور ایجنٹ نے *بالکل وہی منظور شدہ اقدام* انجام دیا (ویسا ہی ڈائجسٹ، رسید جو منظوری کے ساتھ `receipt_hash` کے ذریعے بندھی ہوئی ہے، سبق کی اپنی چین کنونشن) — جب منظوری کی پالیسی ورژن، کلید، اور میعاد ابھی بھی موجود تھیں، بالکل ایک بار۔ اگر کسی بھی طرف میں تبدیلی ہوتی ہے، تو چین بند ہو جاتی ہے، اور انکار کی وجہ آپ کو بتاتی ہے کہ **کونسی** خاصیت خراب ہوئی: پرانی اتھارٹی بمقابلہ تبدیل شدہ اقدام۔

**ثابت نہیں کرتا:** کہ منظوری کی UI نے انسان کو وہی دکھایا جو وہ سمجھ کر دستخط کر رہا تھا (WYSIWYS اپنی ہی ایک مسئلہ ہے)، کہ کلید کو گھمانے سے پہلے زبردستی یا چوری نہیں کیا گیا، یا کہ نیچے کے اثرات اقدام کے مطابق تھے۔ دستخط شدہ ≠ مجاز: پرانی پالیسی پر درست دستخط، گھمایا ہوا کلید، ختم شدہ مدت، یا مختلف ڈائجسٹ یہاں کچھ بھی نہیں دیتا۔

دونوں اقسام کی رسیدیں سبق کے لفافے اور ایک `verify_chain` کوڈ راستہ مشترک رکھتی ہیں: آپ نے جو بندھن مرکزی نوٹ بک میں اقدام رسیدوں کے لیے بنایا ہے وہی کوڈ ہے جو انسان کی منظوری کی جانچ کرتا ہے۔ ایک ویریفائر معاہدہ، الگ الگ pinned اتھارٹیز، قانونی اقدام کے ڈائجسٹ سے جڑے ہوئے اور کچھ نہیں۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
